In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import os


In [8]:
#setting the default paths for the dataset
bas_dir=r"C:\Users\Abcom\OneDrive\Desktop\PYTHON AND ML\VIRTUAL KEYBOARD\supervised learning data"
test_dir=r"C:\Users\Abcom\OneDrive\Desktop\PYTHON AND ML\VIRTUAL KEYBOARD\supervised learning data\chest_xray\test"
train_dir=r"C:\Users\Abcom\OneDrive\Desktop\PYTHON AND ML\VIRTUAL KEYBOARD\supervised learning data\chest_xray\train"
val_dir=r"C:\Users\Abcom\OneDrive\Desktop\PYTHON AND ML\VIRTUAL KEYBOARD\supervised learning data\chest_xray\val"

In [9]:
#displaying the data
print("Train folders:", os.listdir(train_dir))
print("Validation folders:", os.listdir(val_dir))
print("Test folders:", os.listdir(test_dir))

Train folders: ['NORMAL', 'PNEUMONIA']
Validation folders: ['NORMAL', 'PNEUMONIA']
Test folders: ['NORMAL', 'PNEUMONIA']


In [10]:
#LOADING THE DATA
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = 224
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary'
)


C:\Users\Abcom\AppData\Roaming\Python\Python313\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.


In [11]:
#CNN Algorithm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

model = Sequential([
    Input(shape=(224, 224, 3)),#inputSize,RGB

    Conv2D(32, (3,3), activation='relu'),#learns non-linear patterns
    MaxPooling2D(2,2),#removes unnecessary dimensions

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),#2D->1D
    Dense(128, activation='relu'),#interprets the details
    Dropout(0.5),#removes overfitting content,keeps data managed
    Dense(1, activation='sigmoid')#logistic Regression
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']#defines the accuracy and the crossentropy
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#time to train the model
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)


Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 393s 2s/step - accuracy: 0.8119 - loss: 0.4264 - val_accuracy: 0.8750 - val_loss: 0.3623
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 310s 2s/step - accuracy: 0.8802 - loss: 0.2809 - val_accuracy: 0.5625 - val_loss: 0.6959
Epoch 3/10
 99/163 ━━━━━━━━━━━━━━━━━━━━ 2:13 2s/step - accuracy: 0.9004 - loss: 0.2443

In [ ]:
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.show()


In [ ]:
#Shape of data
images, labels = train_data.next()
print(images.shape)
print(labels.shape)
print(train_data.samples)
print(val_data.samples)
#SHAPE (32, 224, 224, 3)

In [ ]:
model.save("xray_model.h5")
